# 02 · Training: U-Net vs. DINOv2 vs. DINOv3 across biomes

**Question.** Forest/non-forest segmentation of Sentinel-2 images saturates inside a single region (IoU ≈ 0.95 on the Amazon), so it cannot
separate encoders. Transfer to another biome does not: models trained on the Amazon reach a weighted IoU of only ≈ 0.81 on the
Atlantic Forest (John & Zhang, 2022). **Do self-supervised foundation-model encoders (DINOv2, DINOv3 pretrained on web or
satellite images) transfer better than an ImageNet U-Net?**

**This notebook only trains and evaluates.** Every run writes two CSV files to `results/raw/`:
- `runs/<run>.csv`: one row with the run's settings, time and parameter counts;
- `per_image/<run>.csv`: the confusion counts (TP, FP, FN, TN) of every evaluated image.

All tables, statistics and figures are made in `03_analysis.ipynb`, which only reads these CSV files. Nothing in the analysis can
influence training, and every metric (IoU, weighted IoU, per-image F1, …) can be recomputed from the counts.

**Rules that keep the transfer test honest**
1. A model only ever sees its *training biome*: training images, and its validation images for choosing the best checkpoint.
2. Input standardization uses statistics of the training biome's training split only.
3. The other biome is used once, at the very end, for evaluation.

**Data:** Bragagnolo et al. (2021), *Amazon and Atlantic Forest image datasets for semantic segmentation*, Zenodo
[10.5281/zenodo.4498086](https://doi.org/10.5281/zenodo.4498086), CC-BY-4.0. Sentinel-2 L2A, bands 4-3-2-8 (R, G, B, NIR),
512 × 512 px, official split (Amazon 499/100/20, Atlantic Forest 485/100/20 train/val/test). Prepare it with `01_data.ipynb`.

## 1. Settings

Everything that changes between runs is read from environment variables, so the same notebook runs on Colab, a laptop or a rented
GPU (`jupyter nbconvert --execute`). Runs that already have a result file are skipped, so an interrupted job can simply be restarted.

In [ ]:
import os, sys, json, math, time, random, platform
from pathlib import Path

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # deterministic matrix products on the GPU (set before CUDA starts)
import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from transformers import Dinov2Model, Dinov2Config, DINOv3ViTModel, DINOv3ViTConfig
from PIL import Image

SMOKE_TEST = os.environ.get("SMOKE_TEST", "0") == "1"   # tiny random models and a few steps: checks that everything runs

def env_list(name, default):
    return [s.strip() for s in os.environ.get(name, default).split(",") if s.strip()]

ALL_MODELS = ["unet_ft", "unet_pft", "dinov2_ft", "dinov2_pft", "dinov3_ft", "dinov3_pft", "dinov3l_pft", "dinov3sat_pft"]
CONFIG = dict(
    data_root=Path(os.environ.get("DATA_ROOT", "../data/bragagnolo")),
    results_dir=Path(os.environ.get("RESULTS_DIR", "../results")),
    models=env_list("MODELS", ",".join(ALL_MODELS)),
    seeds=[int(s) for s in env_list("SEEDS", "0,1,2,3,4")],
    train_biomes=env_list("TRAIN_BIOMES", "amazon,atlantic"),
    epochs=int(os.environ.get("EPOCHS", "40")),      # one random 256-px crop per training image and epoch
    n_evals=15,                                      # validation checks per training (best checkpoint by validation mIoU)
    crop=256, batch_size=16,
    lr_decoder=3e-4, lr_resnet=3e-4,                 # decoder, new layers, ResNet-34
    lr_vit=2e-5, lr_vit_first=1e-4,                  # ViT encoder and its first (patch-embedding) layer
    vit_partial_blocks=4,                            # partial fine-tuning: last 4 of 12 blocks (scaled with depth: 8 of 24)
    pyramid_blocks=(3, 6, 9, 12),                    # ViT blocks that feed the decoder (scaled with depth)
    num_workers=int(os.environ.get("NUM_WORKERS", "4")),
    save_ckpt=os.environ.get("SAVE_CKPT", "1") == "1",
    # pretrained encoders: Hugging Face ids or local folders with the same files
    dinov2_name=os.environ.get("DINOV2_NAME", "facebook/dinov2-small"),
    dinov3_name=os.environ.get("DINOV3_NAME", "facebook/dinov3-vits16-pretrain-lvd1689m"),
    dinov3l_name=os.environ.get("DINOV3L_NAME", "facebook/dinov3-vitl16-pretrain-lvd1689m"),
    dinov3sat_name=os.environ.get("DINOV3SAT_NAME", "facebook/dinov3-vitl16-pretrain-sat493m"),
)
if SMOKE_TEST:
    CONFIG.update(epochs=1, n_evals=2, results_dir=Path(os.environ.get("RESULTS_DIR", "../results_smoke")))
HF_TOKEN = os.environ.get("HF_TOKEN")   # needed only for the gated DINOv3 repositories on Hugging Face
MAX_IMAGES = int(os.environ.get("MAX_IMAGES", "0"))   # >0: use only that many images per split (smoke tests)

RAW = CONFIG["results_dir"] / "raw"
for d in ("runs", "per_image", "history", "ckpt"):
    (RAW / d).mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else platform.processor() or DEVICE.type
print("PyTorch", torch.__version__, "| device:", DEVICE, GPU_NAME)
print({k: str(v) for k, v in CONFIG.items()})

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

# Deterministic mode: same seed + same GPU type + same library versions = same result
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

MODEL_NAMES = {"unet_ft": "U-Net ResNet-34, full", "unet_pft": "U-Net ResNet-34, partial",
               "dinov2_ft": "DINOv2 ViT-S/14, full", "dinov2_pft": "DINOv2 ViT-S/14, partial",
               "dinov3_ft": "DINOv3 ViT-S/16, full", "dinov3_pft": "DINOv3 ViT-S/16, partial",
               "dinov3l_pft": "DINOv3 ViT-L/16 web, partial", "dinov3sat_pft": "DINOv3 ViT-L/16 satellite, partial"}
assert set(CONFIG["models"]) <= set(MODEL_NAMES), CONFIG["models"]

## 2. Data

All images of both biomes are loaded once into memory as 16-bit integers (about 2.5 GB). Masks: 1 = forest, 0 = non-forest.
File names keep the Sentinel-2 tile and date, which `01_data.ipynb` uses to describe the splits.

In [ ]:
BIOMES = {"amazon": "AMAZON", "atlantic": "ATLANTIC FOREST"}
SPLITS = {"train": "Training", "val": "Validation", "test": "Test"}

def list_pairs(biome, split):
    """(image, mask) file pairs of one split, matched by file name. Image folders are image/ or images/, mask folders mask(s)/ or label/."""
    root = CONFIG["data_root"] / BIOMES[biome] / SPLITS[split]
    img_dir = next(d for d in root.iterdir() if d.is_dir() and d.name.lower().startswith("image"))
    msk_dir = next(d for d in root.iterdir() if d.is_dir() and not d.name.lower().startswith("image"))
    masks = {p.stem: p for p in msk_dir.iterdir() if p.suffix.lower() in (".tif", ".tiff", ".png")}
    pairs = [(p, masks[p.stem]) for p in sorted(img_dir.glob("*.tif")) if p.stem in masks]
    return pairs[:MAX_IMAGES] if MAX_IMAGES else pairs

def read_pair(img_path, msk_path):
    with rasterio.open(img_path) as r:
        x = r.read().astype(np.uint16)                       # (4, 512, 512): R, G, B, NIR
    m = np.array(Image.open(msk_path))
    m = (m[..., 0] if m.ndim == 3 else m) > 0
    return x, m.astype(np.uint8)

DATA = {}
t0 = time.time()
for b in BIOMES:
    for s in SPLITS:
        pairs = list_pairs(b, s)
        xs, ms = zip(*(read_pair(*p) for p in pairs))
        DATA[(b, s)] = dict(x=np.stack(xs), m=np.stack(ms), names=[p[0].stem for p in pairs])
        print(f"{b:9s} {s:5s} {len(pairs):4d} images, forest fraction {DATA[(b, s)]['m'].mean():.3f}")
print(f"loaded in {time.time() - t0:.0f} s")

def norm_stats(biome):
    """Per-band mean and standard deviation of the training split of one biome."""
    x = DATA[(biome, "train")]["x"].astype(np.float64)
    return x.mean((0, 2, 3)).astype(np.float32), x.std((0, 2, 3)).astype(np.float32)

class CropDataset(Dataset):
    """Training: one random crop per image and epoch, with a random rotation (0/90/180/270°) and flip. Random numbers come from
    torch, which the DataLoader seeds per worker from the run's generator, so the sequence is reproducible."""
    def __init__(self, x, m, mean, std, crop):
        self.x, self.m, self.crop = x, m, crop
        self.mean, self.std = torch.from_numpy(mean)[:, None, None], torch.from_numpy(std)[:, None, None]
    def __len__(self):
        return len(self.x)
    def __getitem__(self, i):
        H = self.x.shape[-1]; c = self.crop
        r, q = torch.randint(0, H - c + 1, (2,)).tolist()
        img = (torch.from_numpy(self.x[i, :, r:r + c, q:q + c].astype(np.float32)) - self.mean) / self.std
        lab = torch.from_numpy(self.m[i, r:r + c, q:q + c].astype(np.float32))
        k = int(torch.randint(0, 4, (1,)))
        img, lab = torch.rot90(img, k, (1, 2)), torch.rot90(lab, k, (0, 1))
        if torch.rand(1).item() < 0.5:
            img, lab = img.flip(2), lab.flip(1)
        return img.contiguous(), lab.contiguous()

def train_loader(biome, mean, std, seed):
    d = DATA[(biome, "train")]
    return DataLoader(CropDataset(d["x"], d["m"], mean, std, CONFIG["crop"]), batch_size=CONFIG["batch_size"], shuffle=True,
                      drop_last=True, num_workers=CONFIG["num_workers"], pin_memory=DEVICE.type == "cuda",
                      generator=torch.Generator().manual_seed(seed), persistent_workers=CONFIG["num_workers"] > 0,
                      multiprocessing_context="fork" if CONFIG["num_workers"] > 0 and sys.platform != "win32" else None)

## 3. Models

Every model is the same U-Net decoder on top of a different pretrained encoder, with 4 input bands (R, G, B, NIR):

| key | encoder | pretraining | parameters trained |
|---|---|---|---|
| `unet_ft` / `unet_pft` | ResNet-34 | ImageNet (supervised) | all / first layer + stage 4 |
| `dinov2_ft` / `dinov2_pft` | DINOv2 ViT-S/14 | LVD-142M web images (self-supervised) | all / first layer + last 4 blocks |
| `dinov3_ft` / `dinov3_pft` | DINOv3 ViT-S/16 | LVD-1689M web images (self-supervised) | all / first layer + last 4 blocks |
| `dinov3l_pft` | DINOv3 ViT-L/16 | LVD-1689M web images | first layer + last 8 blocks |
| `dinov3sat_pft` | DINOv3 ViT-L/16 | SAT-493M satellite images (Maxar) | first layer + last 8 blocks |

The ViT encoders are turned into a 5-level feature pyramid (UNETR style) so that they can feed the U-Net decoder. The fourth band
(NIR) is added to the pretrained first layer by copying the mean of the RGB filters. The code below is taken from our earlier
benchmark notebook, with the Sentinel-1 branch removed.

In [ ]:
class Resize(nn.Module):
    """Bilinear resizing of a square map as two matrix products; same result as F.interpolate(mode="bilinear"), but with a
    deterministic gradient on the GPU."""
    def __init__(self, n_in, n_out):
        super().__init__()
        m = F.interpolate(torch.eye(n_in)[None], size=n_out, mode="linear", align_corners=False)[0].T
        self.register_buffer("m", m.contiguous())
    def forward(self, x):
        m = self.m.to(x.dtype)
        return m @ x @ m.T

def conv_bn_relu(c_in, c_out, k=3, stride=1):
    return nn.Sequential(nn.Conv2d(c_in, c_out, k, stride, k // 2, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))

def adapt_first_layer(conv, n_in):
    """RGB-pretrained first layer → n_in channels: RGB filters kept, extra channels get the RGB mean, all scaled by 3 / n_in."""
    w = conv.weight.data
    new_w = torch.cat([w, w.mean(1, keepdim=True).repeat(1, n_in - 3, 1, 1)], 1)
    layer = nn.Conv2d(n_in, conv.out_channels, conv.kernel_size, conv.stride, conv.padding, bias=conv.bias is not None)
    layer.weight.data.copy_(new_w * 3 / n_in)
    if conv.bias is not None:
        layer.bias.data.copy_(conv.bias.data)
    return layer

class UNetDecoder(nn.Module):
    """Common decoder: 5 upsampling steps from 1/32 to full size, joining the encoder map of the same size at each step."""
    def __init__(self, enc_ch=(64, 64, 128, 256, 512), dec_ch=(256, 128, 64, 32, 16)):
        super().__init__()
        skips = list(enc_ch[:-1][::-1]) + [0]
        ins = [enc_ch[-1]] + list(dec_ch[:-1])
        self.blocks = nn.ModuleList([nn.Sequential(conv_bn_relu(i + s, o), conv_bn_relu(o, o)) for i, s, o in zip(ins, skips, dec_ch)])
        self.out_channels = dec_ch[-1]
    def forward(self, feats):
        x, skips = feats[-1], feats[:-1][::-1] + [None]
        for block, skip in zip(self.blocks, skips):
            x = F.interpolate(x, scale_factor=2, mode="nearest")
            if skip is not None:
                x = torch.cat([x, skip], 1)
            x = block(x)
        return x

class ResNetEncoder(nn.Module):
    """ResNet-34 pretrained on ImageNet; returns its 5 stages (1/2 … 1/32)."""
    lr_encoder, lr_first = CONFIG["lr_resnet"], CONFIG["lr_resnet"]
    def __init__(self, n_in):
        super().__init__()
        self.net = smp.encoders.get_encoder("resnet34", in_channels=3, weights=None if SMOKE_TEST else "imagenet")
        self.net.conv1 = adapt_first_layer(self.net.conv1, n_in)
    @property
    def first(self):
        return self.net.conv1
    @property
    def pretrained(self):
        return self.net
    def freeze_for_partial(self):
        """Partial fine-tuning: first layer and stage 4 are trained (3 of 16 residual blocks, ≈ 60 % of the encoder's parameters)."""
        self.frozen = [self.net.bn1, self.net.layer1, self.net.layer2, self.net.layer3]
        for module in self.frozen:
            for p in module.parameters():
                p.requires_grad = False
    def train(self, mode=True):
        super().train(mode)
        for module in getattr(self, "frozen", []):
            module.eval()              # frozen BatchNorm keeps its ImageNet statistics
        return self
    def forward(self, x):
        return self.net(x)[1:]

from torch.nn.attention import sdpa_kernel, SDPBackend
if hasattr(torch.backends.cuda, "allow_fp16_bf16_reduction_math_sdp"):
    torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)

def load_vit(family):
    attn = "sdpa"
    if family == "dinov2":
        if SMOKE_TEST:
            return Dinov2Model(Dinov2Config(hidden_size=96, num_hidden_layers=12, num_attention_heads=3, intermediate_size=384,
                                            patch_size=14, image_size=224, attn_implementation=attn))
        return Dinov2Model.from_pretrained(CONFIG["dinov2_name"], attn_implementation=attn)
    if SMOKE_TEST:   # the ViT-L families get 24 layers so that the depth-dependent logic is exercised
        return DINOv3ViTModel(DINOv3ViTConfig(hidden_size=96, num_hidden_layers=12 if family == "dinov3" else 24, num_attention_heads=3,
                                              intermediate_size=384, patch_size=16, num_register_tokens=4, attn_implementation=attn))
    return DINOv3ViTModel.from_pretrained(CONFIG[f"{family}_name"], token=HF_TOKEN, attn_implementation=attn)

class ViTEncoder(nn.Module):
    """DINOv2 / DINOv3 (ViT-S or ViT-L) as 5 feature maps for the U-Net decoder (UNETR-style pyramid)."""
    lr_encoder, lr_first = CONFIG["lr_vit"], CONFIG["lr_vit_first"]
    def __init__(self, family, n_in, size):
        super().__init__()
        vit = load_vit(family)
        cfg = vit.config
        self.patch = cfg.patch_size
        L = cfg.num_hidden_layers                                       # 12 (ViT-S) or 24 (ViT-L)
        self.pyramid = [b * L // 12 for b in CONFIG["pyramid_blocks"]]
        self.n_partial = CONFIG["vit_partial_blocks"] * L // 12
        self.input_size = (size // self.patch) * self.patch             # DINOv2: 256 → 252 (multiple of 14)
        self.grid = self.input_size // self.patch
        self.n_prefix = 1 + getattr(cfg, "num_register_tokens", 0)      # CLS (+ 4 register tokens in DINOv3)
        emb = vit.embeddings
        if family == "dinov2":
            with torch.no_grad():   # adapt the learned position embeddings to the grid once
                pos = emb.interpolate_pos_encoding(torch.zeros(1, 1 + self.grid ** 2, cfg.hidden_size), self.input_size, self.input_size)
            emb.position_embeddings = nn.Parameter(pos.detach().clone())
            emb.patch_embeddings.projection = adapt_first_layer(emb.patch_embeddings.projection, n_in)
            emb.patch_embeddings.num_channels = n_in
        else:                       # DINOv3: rotary position embeddings, valid for any grid
            emb.patch_embeddings = adapt_first_layer(emb.patch_embeddings, n_in)
        cfg.num_channels = n_in
        self.vit = vit
        dim, G = cfg.hidden_size, size // 16
        self.to_G = Resize(self.grid, G) if self.grid != G else nn.Identity()
        self.level4 = nn.Sequential(Resize(G, 2 * G), conv_bn_relu(dim, 128), Resize(2 * G, 4 * G), conv_bn_relu(128, 64))
        self.level8 = nn.Sequential(Resize(G, 2 * G), conv_bn_relu(dim, 128))
        self.level16 = conv_bn_relu(dim, 256, k=1)
        self.level32 = conv_bn_relu(dim, 512, stride=2)
        self.level2 = nn.Sequential(conv_bn_relu(n_in, 32, stride=2), conv_bn_relu(32, 64))
    @property
    def pretrained(self):
        return self.vit
    @property
    def first(self):
        pe = self.vit.embeddings.patch_embeddings
        return pe.projection if hasattr(pe, "projection") else pe
    @property
    def blocks(self):
        return next(m for m in self.vit.modules() if isinstance(m, nn.ModuleList) and len(m) == self.vit.config.num_hidden_layers)
    @property
    def final_norm(self):
        return [m for name, m in self.vit.named_children() if "norm" in name][-1]
    def freeze_for_partial(self):
        """Partial fine-tuning: only the first layer, the last blocks and the final normalization are trained."""
        keep = {id(p) for m in [self.first, self.final_norm, *self.blocks[-self.n_partial:]] for p in m.parameters()}
        for p in self.vit.parameters():
            p.requires_grad = id(p) in keep
    def forward(self, x):
        b = x.shape[0]
        xin = x if self.input_size == x.shape[-1] else F.interpolate(x, size=(self.input_size,) * 2, mode="bilinear", align_corners=False)
        with sdpa_kernel(SDPBackend.MATH):
            out = self.vit(pixel_values=xin, output_hidden_states=True)
        chosen = [out.hidden_states[i] for i in self.pyramid[:-1]] + [out.last_hidden_state]
        maps = [self.to_G(h[:, self.n_prefix:].transpose(1, 2).reshape(b, -1, self.grid, self.grid)) for h in chosen]
        return [self.level2(x), self.level4(maps[0]), self.level8(maps[1]), self.level16(maps[2]), self.level32(maps[3])]

class SegModel(nn.Module):
    def __init__(self, family, n_in=4, size=CONFIG["crop"]):
        super().__init__()
        self.encoder = ResNetEncoder(n_in) if family == "unet" else ViTEncoder(family, n_in, size)
        self.decoder = UNetDecoder()
        self.head = nn.Conv2d(self.decoder.out_channels, 1, 3, padding=1)
    def forward(self, x):
        return self.head(self.decoder(self.encoder(x)))

def build_model(kind):
    family, mode = kind.split("_")
    model = SegModel(family)
    if mode == "pft":
        model.encoder.freeze_for_partial()
    return model

def make_optimizer(model):
    """Three learning rates: pretrained encoder, its first layer, and everything new (decoder and adapters)."""
    enc = model.encoder
    first = [p for p in enc.first.parameters() if p.requires_grad]
    used = {id(p) for p in first}
    body = [p for p in enc.pretrained.parameters() if p.requires_grad and id(p) not in used]
    used |= {id(p) for p in body}
    rest = [p for p in model.parameters() if p.requires_grad and id(p) not in used]
    groups = [{"params": first, "lr": enc.lr_first}, {"params": body, "lr": enc.lr_encoder}, {"params": rest, "lr": CONFIG["lr_decoder"]}]
    return torch.optim.AdamW([g for g in groups if g["params"]], weight_decay=1e-4)

def loss_fn(logits, t):
    """Binary cross-entropy + Dice."""
    bce = F.binary_cross_entropy_with_logits(logits, t)
    p = torch.sigmoid(logits)
    return bce + 1 - (2 * (p * t).sum() + 1) / (p.sum() + t.sum() + 1)

## 4. Training and evaluation

- **Training:** AdamW with a cosine schedule, mixed precision on GPU, `epochs` × (training images / batch) steps.
- **Model selection:** 15 checks on the training biome's validation split; the checkpoint with the best validation mIoU is kept.
- **Learning curves:** at every check, `history/<run>.csv` records the learning rate, the training loss and training metrics
  (pooled over the augmented training crops since the previous check, model in training mode) and the validation loss and metrics
  (mIoU, IoU per class, forest F1, precision, recall, pixel accuracy). `03_analysis.ipynb` plots them to check convergence.
- **Evaluation:** whole 512 × 512 images, cut into four 256 × 256 tiles (the size the models were trained on). Evaluated sets:
  the official test split of **both** biomes, and the **whole other biome** (train + val + test, 605 images), which the model has
  never seen and which gives a much larger transfer test set than the 20 official test images.
- **Checkpoints** (`SAVE_CKPT=1`): only the trained parameters and all buffers, in `safetensors` format. The frozen parts are the
  published pretrained weights and are not repeated.

In [ ]:
N_BINS = 20        # probability histogram bins per class (threshold-free metrics: ROC/AUC, average precision, calibration)
BOUNDARY_TOL = 2   # pixels of tolerance for the boundary F1

def boundary(mask):
    """Inner boundary pixels of a boolean map (N, H, W): pixels with a different-class 4-neighbour."""
    f = mask.float()[:, None]
    dil = F.max_pool2d(f, 3, 1, 1); ero = -F.max_pool2d(-f, 3, 1, 1)
    return (dil != ero)[:, 0] & mask

def extra_stats(logits, t):
    """Per-image probability histograms (true forest / true non-forest pixels) and boundary-match counts."""
    p = torch.sigmoid(logits)
    bins = torch.clamp((p * N_BINS).long(), max=N_BINS - 1)
    n = logits.shape[0]
    idx = bins.reshape(n, -1) + N_BINS * t.reshape(n, -1).long()          # 0..19 non-forest, 20..39 forest
    hist = torch.zeros(n, 2 * N_BINS, dtype=torch.long, device=logits.device).scatter_add_(1, idx, torch.ones_like(idx))
    pred = logits > 0
    k = 2 * BOUNDARY_TOL + 1
    bp, bt = boundary(pred) | boundary(~pred), boundary(t) | boundary(~t)
    near_t = F.max_pool2d(bt.float()[:, None], k, 1, BOUNDARY_TOL)[:, 0] > 0
    near_p = F.max_pool2d(bp.float()[:, None], k, 1, BOUNDARY_TOL)[:, 0] > 0
    b = torch.stack([(bp & near_t).sum((1, 2)), bp.sum((1, 2)), (bt & near_p).sum((1, 2)), bt.sum((1, 2))], 1)
    return torch.cat([hist, b], 1).cpu().numpy()

EXTRA_COLS = [f"hist_nonforest_{i:02d}" for i in range(N_BINS)] + [f"hist_forest_{i:02d}" for i in range(N_BINS)] + \
             ["bnd_pred_matched", "bnd_pred", "bnd_true_matched", "bnd_true"]

@torch.no_grad()
def predict(model, x_u16, mean, std, masks=None, bs=8, extra=False):
    """Forest predictions for whole images (N, 4, 512, 512), as 2 × 2 tiles of 256 px.
    Returns the boolean predictions, the mean loss over the batches (if masks are given) and, with extra=True,
    the per-image extra statistics of extra_stats()."""
    model.eval()
    c = CONFIG["crop"]; out, losses, extras = [], [], []
    m_t, s_t = torch.from_numpy(mean)[:, None, None].to(DEVICE), torch.from_numpy(std)[:, None, None].to(DEVICE)
    for k in range(0, len(x_u16), bs):
        x = (torch.from_numpy(x_u16[k:k + bs].astype(np.float32)).to(DEVICE) - m_t) / s_t
        n, _, H, W = x.shape
        tiles = x.unfold(2, c, c).unfold(3, c, c)                       # n, C, H/c, W/c, c, c
        gh, gw = tiles.shape[2], tiles.shape[3]
        tiles = tiles.permute(0, 2, 3, 1, 4, 5).reshape(-1, x.shape[1], c, c)
        with torch.autocast(DEVICE.type, enabled=USE_AMP):
            logits = model(tiles)[:, 0].float()
        logits = logits.reshape(n, gh, gw, c, c).permute(0, 1, 3, 2, 4).reshape(n, H, W)
        if masks is not None:
            t = torch.from_numpy(masks[k:k + bs].astype(np.float32)).to(DEVICE)
            losses.append(loss_fn(logits, t).item())
            if extra:
                extras.append(extra_stats(logits, t > 0.5))
        out.append((logits > 0).cpu().numpy())
    loss = float(np.mean(losses)) if losses else None
    if extra:
        return np.concatenate(out), loss, np.concatenate(extras)
    return np.concatenate(out), loss

def confusion(pred, m):
    """Per-image TP, FP, FN, TN (forest = positive)."""
    t = m.astype(bool)
    return np.stack([(pred & t).sum((1, 2)), (pred & ~t).sum((1, 2)), (~pred & t).sum((1, 2)), (~pred & ~t).sum((1, 2))], 1)

def scores(c):
    """Pooled metrics from confusion counts (rows = images or batches)."""
    tp, fp, fn, tn = c.sum(0).astype(np.float64)
    iou_f, iou_n = tp / max(tp + fp + fn, 1), tn / max(tn + fn + fp, 1)
    return dict(miou=(iou_f + iou_n) / 2, iou_forest=iou_f, iou_nonforest=iou_n,
                f1_forest=2 * tp / max(2 * tp + fp + fn, 1), acc=(tp + tn) / max(tp + fp + fn + tn, 1),
                precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1))

def eval_sets(train_biome):
    other = next(b for b in BIOMES if b != train_biome)
    sets = {f"{b}_test": [(b, "test")] for b in BIOMES}
    sets[f"{other}_all"] = [(other, s) for s in SPLITS]
    return sets

def trainable_state(model):
    """Trained parameters + all buffers (BatchNorm statistics, resize matrices)."""
    keep = {n for n, p in model.named_parameters() if p.requires_grad} | {n for n, _ in model.named_buffers()}
    return {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k in keep}

def run_id(kind, seed, train_biome):
    return f"{kind}__{train_biome}__s{seed}"

def train_one(kind, seed, train_biome):
    rid = run_id(kind, seed, train_biome)
    set_seed(seed)
    mean, std = norm_stats(train_biome)
    dl = train_loader(train_biome, mean, std, seed)
    val = DATA[(train_biome, "val")]
    steps = CONFIG["epochs"] * len(dl)
    eval_every = max(1, math.ceil(steps / CONFIG["n_evals"]))

    model = build_model(kind).to(DEVICE)
    opt = make_optimizer(model)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    best, best_state, best_step, step, bad, t0 = -1.0, None, 0, 0, 0, time.time()
    hist, run_loss, n_loss, run_c = [], 0.0, 0, np.zeros(4, np.int64)
    while step < steps:
        for x, y in dl:
            model.train()
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(DEVICE.type, enabled=USE_AMP):
                logits = model(x)[:, 0]
            loss = loss_fn(logits.float(), y)
            if not torch.isfinite(loss):
                bad += 1
                if bad >= 20:
                    raise RuntimeError(f"{rid}: loss not finite in 20 consecutive steps; training diverged.")
            else:
                bad = 0; run_loss += loss.item(); n_loss += 1
                with torch.no_grad():   # training-batch confusion counts (augmented crops, model in train mode)
                    p, t = logits.detach() > 0, y > 0.5
                    run_c += np.array([(p & t).sum().item(), (p & ~t).sum().item(), (~p & t).sum().item(), (~p & ~t).sum().item()])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            step += 1
            if step % eval_every == 0 or step == steps:
                pred, val_loss = predict(model, val["x"], mean, std, masks=val["m"])
                vs, ts = scores(confusion(pred, val["m"])), scores(run_c[None])
                v = vs["miou"]
                hist.append(dict(step=step, epoch=round(step / len(dl), 3), lr=sched.get_last_lr()[0],
                                 train_loss=run_loss / max(n_loss, 1), **{f"train_{k}": x for k, x in ts.items()},
                                 val_loss=val_loss, **{f"val_{k}": x for k, x in vs.items()},
                                 elapsed_s=round(time.time() - t0, 1)))
                run_loss, n_loss, run_c = 0.0, 0, np.zeros(4, np.int64)
                if v > best:
                    best, best_step = v, step
                    best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
            if step >= steps:
                break
    train_time = time.time() - t0
    model.load_state_dict(best_state)

    rows = []
    for set_name, parts in eval_sets(train_biome).items():
        for part in parts:
            d = DATA[part]
            pred, img_loss, ex = predict(model, d["x"], mean, std, masks=d["m"], extra=True)
            c = confusion(pred, d["m"])
            for name, (tp, fp, fn, tn), e in zip(d["names"], c, ex):
                rows.append(dict(run=rid, model=kind, seed=seed, train_biome=train_biome, eval_set=set_name,
                                 biome=part[0], split=part[1], image=name, tp=tp, fp=fp, fn=fn, tn=tn,
                                 **dict(zip(EXTRA_COLS, e.tolist()))))
    if CONFIG["save_ckpt"]:
        from safetensors.torch import save_file
        save_file(trainable_state(model), str(RAW / "ckpt" / f"{rid}.safetensors"))
    pd.DataFrame(hist).to_csv(RAW / "history" / f"{rid}.csv", index=False)
    pd.DataFrame(rows).to_csv(RAW / "per_image" / f"{rid}.csv", index=False)
    run = dict(run=rid, model=kind, seed=seed, train_biome=train_biome, epochs=CONFIG["epochs"], steps=steps, best_step=best_step,
               val_miou=best, train_time_s=round(train_time, 1), eval_time_s=round(time.time() - t0 - train_time, 1),
               total_params=sum(p.numel() for p in model.parameters()),
               trainable_params=sum(p.numel() for p in model.parameters() if p.requires_grad),
               gpu=GPU_NAME, torch=torch.__version__, smoke_test=SMOKE_TEST)
    pd.DataFrame([run]).to_csv(RAW / "runs" / f"{rid}.csv", index=False)   # written last: marks the run as complete
    return run

## 5. Run everything

Order: seed by seed, so that an interrupted job leaves complete seeds (every model, both training biomes) rather than some models
with more seeds than others. The analysis only uses seeds that are complete for all models.

In [ ]:
todo = [(k, s, b) for s in CONFIG["seeds"] for b in CONFIG["train_biomes"] for k in CONFIG["models"]]
print(len(todo), "runs selected;", sum((RAW / "runs" / f"{run_id(*t)}.csv").exists() for t in todo), "already done")
for kind, seed, biome in todo:
    if (RAW / "runs" / f"{run_id(kind, seed, biome)}.csv").exists():
        continue
    r = train_one(kind, seed, biome)
    print(f"{r['run']:32s} val mIoU {r['val_miou']:.4f} (best step {r['best_step']}/{r['steps']}), "
          f"{r['train_time_s'] / 60:.1f} min train, {r['eval_time_s']:.0f} s eval", flush=True)
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
print("done")